In [13]:
import os, glob, random, math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


DATA_ROOT = "./datasets"


CITIES = ["tunis", "manila", "copenhagen"]
YEARS_TRAIN_SEQ = [2014, 2020, 2025]  

# Your fine-tuned SegFormer checkpoint (HF folder or model id)
SEGFORMER_CKPT = "./models/segformer_osm_esri_grouped/final"


NUM_CLASSES = 7

# Tile size expected (your images should be consistent)
IMG_SIZE = 256   # 256 recommended (faster). You can use 512 if GPU allows.

# Training
BATCH_SIZE = 8
EPOCHS = 25
LR = 2e-4
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


Device: cuda


In [15]:
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
segformer = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_CKPT).to(device)
segformer.eval()

# Freeze SegFormer (we only train the temporal model)
for p in segformer.parameters():
    p.requires_grad = False

@torch.no_grad()
def segformer_predict_mask(pil_img: Image.Image) -> torch.Tensor:
    """
    Returns predicted class-id mask: (H, W) torch.long on CPU
    """
    # Ensure consistent size
    pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)

    inputs = processor(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = segformer(**inputs)
    logits = outputs.logits  # (B, C, h, w) usually smaller than IMG_SIZE

    # Upsample to IMG_SIZE
    logits_up = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    pred = torch.argmax(logits_up, dim=1).squeeze(0).to("cpu").long()  # (H, W)

    return pred


c:\Users\2640870\AppData\Local\anaconda3\envs\torch_gpu\lib\site-packages\transformers\image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


In [16]:
import os, glob, random
from typing import Dict, List, Tuple

IMG_EXTS = (".jpeg")

def list_tiles_by_folder(city: str, year: int) -> Dict[str, str]:
    """
    Returns dict: tile_id -> filepath

    Expected structure:
      DATA_ROOT/<city>/<tile_id>/*<year>*.(png/jpg/jpeg/tif/tiff)
    Example:
      tunis/out_12345/ESRI 2025.jpeg
    """
    # search inside each out_*/ folder for an image containing the year in its name
    pattern = os.path.join(DATA_ROOT, city, "out_*", "*")
    files = [f for f in glob.glob(pattern) if f.lower().endswith(IMG_EXTS)]

    out: Dict[str, str] = {}
    year_str = str(year)

    for f in files:
        base = os.path.basename(f)
        if year_str not in base:
            continue

        tile_id = os.path.basename(os.path.dirname(f))  # e.g., out_12345

        # If multiple matches exist, keep the first one; or override deterministically
        # Here: prefer "ESRI" if present, otherwise keep existing.
        if tile_id not in out:
            out[tile_id] = f
        else:
            if "esri" in base.lower() and "esri" not in os.path.basename(out[tile_id]).lower():
                out[tile_id] = f

    return out


def build_triplets() -> List[Tuple[str, str, str, str]]:
    """
    Returns list of triplets: (path_2014, path_2020, path_2025, tile_id)
    Only keeps tiles available in all 3 years.
    """
    triplets: List[Tuple[str, str, str, str]] = []

    for city in CITIES:
        m2014 = list_tiles_by_folder(city, 2014)
        m2020 = list_tiles_by_folder(city, 2020)
        m2025 = list_tiles_by_folder(city, 2025)

        common = sorted(set(m2014) & set(m2020) & set(m2025))
        print(city, "common tiles:", len(common))

        for tid in common:
            triplets.append((m2014[tid], m2020[tid], m2025[tid], tid))

    random.shuffle(triplets)
    return triplets


triplets = build_triplets()
print("Total triplets:", len(triplets))

split = int(0.9 * len(triplets))
train_triplets = triplets[:split]
val_triplets = triplets[split:]
print("Train:", len(train_triplets), "Val:", len(val_triplets))


tunis common tiles: 0
manila common tiles: 0
copenhagen common tiles: 0
Total triplets: 0
Train: 0 Val: 0


In [17]:
class MaskSequenceDataset(Dataset):
    """
    Returns:
      x_seq: (T=2, C, H, W) one-hot masks for [2014, 2020]
      y:     (H, W) class-id mask for 2025
    """
    def __init__(self, triplets, num_classes: int):
        self.triplets = triplets
        self.num_classes = num_classes

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        p2014, p2020, p2025, tid = self.triplets[idx]

        img2014 = Image.open(p2014).convert("RGB")
        img2020 = Image.open(p2020).convert("RGB")
        img2025 = Image.open(p2025).convert("RGB")

        m2014 = segformer_predict_mask(img2014)  # (H, W) long
        m2020 = segformer_predict_mask(img2020)
        m2025 = segformer_predict_mask(img2025)

        # one-hot encode inputs
        oh2014 = F.one_hot(m2014, num_classes=self.num_classes).permute(2,0,1).float()  # (C,H,W)
        oh2020 = F.one_hot(m2020, num_classes=self.num_classes).permute(2,0,1).float()

        x_seq = torch.stack([oh2014, oh2020], dim=0)  # (T=2, C, H, W)
        y = m2025  # (H,W) long

        return x_seq, y

train_ds = MaskSequenceDataset(train_triplets, NUM_CLASSES)
val_ds   = MaskSequenceDataset(val_triplets, NUM_CLASSES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        p = k // 2
        self.hid_ch = hid_ch
        self.conv = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, kernel_size=k, padding=p)

    def forward(self, x, h, c):
        # x: (B, in_ch, H, W)
        # h,c: (B, hid_ch, H, W)
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MaskForecaster(nn.Module):
    """
    Input:  (B, T, C, H, W) one-hot masks for past timesteps
    Output: (B, K, H, W) logits for next mask
    """
    def __init__(self, num_classes, hidden=64):
        super().__init__()
        self.cell = ConvLSTMCell(in_ch=num_classes, hid_ch=hidden, k=3)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, num_classes, 1)
        )

    def forward(self, x_seq):
        B, T, C, H, W = x_seq.shape
        h = torch.zeros((B, self.cell.hid_ch, H, W), device=x_seq.device)
        c = torch.zeros((B, self.cell.hid_ch, H, W), device=x_seq.device)

        for t in range(T):
            x = x_seq[:, t]
            h, c = self.cell(x, h, c)

        logits = self.head(h)
        return logits

model = MaskForecaster(num_classes=NUM_CLASSES, hidden=96).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


In [ ]:
def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0

    for x_seq, y in loader:
        x_seq = x_seq.to(device, non_blocking=True)          # (B,T,C,H,W)
        y = y.to(device, non_blocking=True)                  # (B,H,W)

        logits = model(x_seq)                                # (B,K,H,W)
        loss = criterion(logits, y)

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        bs = x_seq.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / max(1, n)

best_val = float("inf")
for epoch in range(1, EPOCHS+1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "mask_forecaster_best.pt")
        print("  saved: mask_forecaster_best.pt")


In [ ]:
# Load best model
model.load_state_dict(torch.load("mask_forecaster_best.pt", map_location=device))
model.eval()

@torch.no_grad()
def predict_next_mask_from_two_images(img_prev: Image.Image, img_curr: Image.Image) -> np.ndarray:
    """
    Predict next mask given two images (prev, curr) using:
      SegFormer -> masks -> ConvLSTM forecaster -> next mask
    Returns: (H, W) numpy uint8 of class ids
    """
    m_prev = segformer_predict_mask(img_prev)   # (H,W)
    m_curr = segformer_predict_mask(img_curr)

    oh_prev = F.one_hot(m_prev, num_classes=NUM_CLASSES).permute(2,0,1).float()
    oh_curr = F.one_hot(m_curr, num_classes=NUM_CLASSES).permute(2,0,1).float()

    x_seq = torch.stack([oh_prev, oh_curr], dim=0).unsqueeze(0).to(device)  # (1,2,C,H,W)
    logits = model(x_seq)                                                   # (1,K,H,W)
    pred = torch.argmax(logits, dim=1).squeeze(0).to("cpu").numpy().astype(np.uint8)
    return pred

# Example: pick one tile from val_triplets
p2014, p2020, p2025, tid = val_triplets[0]
img2020 = Image.open(p2020).convert("RGB")
img2025 = Image.open(p2025).convert("RGB")

mask2030 = predict_next_mask_from_two_images(img2020, img2025)

print("Predicted 2030 mask shape:", mask2030.shape, "unique classes:", np.unique(mask2030))
